In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
_batch = d.get_toy_databatch(dataset, device)

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78102)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 64389)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 53143)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 50693)

# #### Device() ####
# device = cuda:2

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:2)
# edge_attr                (32464, 16)              Tensor (cuda:2)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:2)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [2]:
from modules.layers import MultiheadSetPooling
from modules.loss import MultiLoss, KLDLoss
from modules.model import BaseAutoencoder
from modules.norm import LogCounts
from modules.train import Loader
from modules.trainers import ReconstrTrainer
from modules.utils import dict_summary

import torch
import torch.nn as nn

In [3]:
loader = Loader(
    dataset,
    device=device,
    batch_size=128,
)

trainer = ReconstrTrainer(
    lr=1e-3, 
    norm_class=LogCounts,
    out_keys={'input':'x_preds', 'target':'x_t', 'mu':'z_mu', 'logvar':'z_logvar'},
    loss_class=MultiLoss,
    loss_kwargs={
        'loss_classes': [nn.MSELoss, KLDLoss],
        # 'ema_norm': True,
        'loss_weights': (1,1e-5),
        # 'warmup':40*7
    }
)

ae = BaseAutoencoder(
    dataset=dataset,
    embed_dim=128,
    num_heads=4,
    method='set',

    norm_class=LogCounts,
    encoder_class=nn.Linear,
    pooling_class=MultiheadSetPooling,
    variational=True,

    hidden_dims=2,
    act_fn=nn.ReLU,
    norm_fn='layer',

    norm_kwargs={'libnorm':True, 'znorm':True, 'learnable':True}
)

trainer.run(
    model=ae,
    loader=loader,
    num_epochs=50,
    report_metrics=['loss','kld','rmse','mae','r2'],
    verbose=True
)

  0%|          | 0/50 [00:00<?, ?it/s]

Test	 loss=0.5114    kld=28.6948    rmse=0.8634    mae=0.5873    r2=0.8058



In [4]:
out = trainer.model(_batch, need_weights=True)
# out = ae(_batch, need_weights=False)
if isinstance(out, torch.Tensor):
    print(out.shape)
elif isinstance(out, dict):
    print(dict_summary(out))
else:
    print(out)



# x_t                      (279872, 1)              Tensor (cuda:2)
# layer_outs               4                        dict
# h_node                   (64, 4373, 128)          Tensor (cuda:2)
# h_pool                   (64, 305, 128)           Tensor (cuda:2)
# z_mu                     (64, 128)                Tensor (cuda:2)
# z_logvar                 (64, 128)                Tensor (cuda:2)
# z                        (64, 128)                Tensor (cuda:2)
# x_preds                  (279872, 1)              Tensor (cuda:2)

